# Notification Service System Design
## Software Engineering Assessment – Part 2

Build a working prototype of a scalable notification service supporting Email, SMS, and Push notifications.

The prototype demonstrates:
- Multiple extensible notification channels
- User preferences and opt-outs
- Idempotency and de-duplication
- Delivery status tracking
- Transactional vs bulk priority
- Retry handling with exponential backoff
- A Flask REST API
- HTML/CSS/JavaScript dashboard
- A SQLite prototype datastore

> **Production note:** SQLite, Python queues, and threads are used for demonstration. At production scale, these would be replaced with durable distributed infrastructure such as PostgreSQL and Kafka/RabbitMQ/SQS.

# 1. Requirements

### Core Functionality
1. Support Email, SMS and Push.
2. Remain extensible to new channels.
3. Respect user preferences and opt-outs.
4. Support per-channel settings.
5. Support quiet-hour configuration.
6. Prevent duplicate notifications using idempotency.
7. Track delivery status.

### Scale & Reliability
- Approximately 10 million notifications/day.
- Handle large traffic spikes.
- At-least-once processing with idempotency.
- Transactional notifications should be near real-time.
- Bulk traffic may be delayed.
- Tolerate unreliable/rate-limited providers.

# 2. High-Level Architecture

```text
Internal Services
       |
       v
Notification API
       |
       v
Validation + Idempotency
       |
       v
Notification Database
       |
       v
Priority Message Queues
       |
   +---+---+
   |       |
   v       v
Transactional   Bulk
Queue           Queue
   |       |
   +---+---+
       |
       v
Notification Workers
       |
  +----+----+
  |    |    |
  v    v    v
Email SMS Push
Providers
```

The key design idea is to keep the API fast and move provider delivery into asynchronous workers.

# 3. Technology Choices

- **Frontend:** HTML, CSS, JavaScript
- **Backend:** Python + Flask
- **Prototype database:** SQLite
- **Prototype queues:** Python `PriorityQueue`
- **Prototype workers:** Python threads

For production:
- PostgreSQL for durable relational storage
- Kafka, RabbitMQ, AWS SQS, or Google Pub/Sub for durable queues
- Horizontally scalable worker services
- Real email/SMS/push provider adapters

In [ ]:
!pip install flask flask-cors

In [ ]:
import sqlite3
import uuid
import time
import random
import threading

from queue import PriorityQueue, Empty
from datetime import datetime
from flask import Flask, request, jsonify
from flask_cors import CORS

# 4. Database Design

The prototype uses three tables:

- `notifications` — the notification and its current status
- `user_preferences` — channel preferences and quiet-hour configuration
- `delivery_attempts` — every delivery attempt, including failures and retries

In [ ]:
connection = sqlite3.connect(
    "notifications.db",
    check_same_thread=False
)
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS notifications (
    id TEXT PRIMARY KEY,
    user_id TEXT NOT NULL,
    channel TEXT NOT NULL,
    message TEXT NOT NULL,
    notification_type TEXT NOT NULL,
    priority INTEGER NOT NULL,
    idempotency_key TEXT UNIQUE NOT NULL,
    status TEXT NOT NULL,
    retry_count INTEGER DEFAULT 0,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS user_preferences (
    user_id TEXT PRIMARY KEY,
    email_enabled INTEGER DEFAULT 1,
    sms_enabled INTEGER DEFAULT 1,
    push_enabled INTEGER DEFAULT 1,
    quiet_start INTEGER,
    quiet_end INTEGER
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS delivery_attempts (
    id TEXT PRIMARY KEY,
    notification_id TEXT NOT NULL,
    attempt_number INTEGER NOT NULL,
    status TEXT NOT NULL,
    error_message TEXT,
    created_at TEXT NOT NULL
)
""")

connection.commit()
print("Database created successfully.")

# 5. User Preferences

Before queueing a notification, the service checks whether the selected channel is enabled for the user.

The schema also stores quiet-hour settings. A production implementation should apply quiet-hour logic according to the product's policy, including whether transactional notifications may bypass quiet hours.

In [ ]:
def create_user_preferences(
    user_id,
    email_enabled=True,
    sms_enabled=True,
    push_enabled=True,
    quiet_start=None,
    quiet_end=None
):
    cursor.execute("""
    INSERT OR REPLACE INTO user_preferences (
        user_id,
        email_enabled,
        sms_enabled,
        push_enabled,
        quiet_start,
        quiet_end
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        user_id,
        int(email_enabled),
        int(sms_enabled),
        int(push_enabled),
        quiet_start,
        quiet_end
    ))
    connection.commit()


def can_send_notification(user_id, channel, notification_type):
    cursor.execute("""
    SELECT email_enabled, sms_enabled, push_enabled,
           quiet_start, quiet_end
    FROM user_preferences
    WHERE user_id = ?
    """, (user_id,))

    preference = cursor.fetchone()

    if preference is None:
        return True

    email_enabled, sms_enabled, push_enabled, quiet_start, quiet_end = preference

    settings = {
        "email": email_enabled,
        "sms": sms_enabled,
        "push": push_enabled
    }

    # Channel opt-out is enforced here.
    if not bool(settings.get(channel, False)):
        return False

    # Quiet-hour fields are stored for the prototype.
    # A production implementation should evaluate the current
    # local time and the product's transactional override policy.
    return True

In [ ]:
create_user_preferences(
    user_id="user_001",
    email_enabled=True,
    sms_enabled=False,
    push_enabled=True
)

print("User preferences created.")

# 6. Priority Queues

Transactional notifications such as password resets should be processed before bulk marketing traffic.

Lower numeric priority means higher priority.

In [ ]:
TRANSACTIONAL_PRIORITY = 1
BULK_PRIORITY = 10

transactional_queue = PriorityQueue()
bulk_queue = PriorityQueue()

# 7. Idempotency and De-duplication

Every request includes an `idempotency_key`.

If the same key is received again, the existing notification is returned instead of creating another notification record.

This is important because at-least-once processing means a request may be retried.

In [ ]:
def create_notification(
    user_id,
    channel,
    message,
    notification_type,
    idempotency_key
):
    cursor.execute("""
    SELECT id, status
    FROM notifications
    WHERE idempotency_key = ?
    """, (idempotency_key,))

    existing = cursor.fetchone()

    if existing:
        return {
            "duplicate": True,
            "notification_id": existing[0],
            "status": existing[1]
        }

    if channel not in {"email", "sms", "push"}:
        return {"error": f"Unsupported channel: {channel}"}

    if notification_type not in {"transactional", "bulk"}:
        return {"error": "notification_type must be transactional or bulk"}

    if not can_send_notification(user_id, channel, notification_type):
        return {
            "error": "User preferences do not allow this notification."
        }

    notification_id = str(uuid.uuid4())

    priority = (
        TRANSACTIONAL_PRIORITY
        if notification_type == "transactional"
        else BULK_PRIORITY
    )

    now = datetime.utcnow().isoformat()

    cursor.execute("""
    INSERT INTO notifications (
        id, user_id, channel, message, notification_type,
        priority, idempotency_key, status, retry_count,
        created_at, updated_at
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        notification_id,
        user_id,
        channel,
        message,
        notification_type,
        priority,
        idempotency_key,
        "QUEUED",
        0,
        now,
        now
    ))

    connection.commit()

    notification = {
        "id": notification_id,
        "user_id": user_id,
        "channel": channel,
        "message": message,
        "notification_type": notification_type,
        "priority": priority
    }

    if priority == TRANSACTIONAL_PRIORITY:
        transactional_queue.put((priority, notification))
    else:
        bulk_queue.put((priority, notification))

    return {
        "duplicate": False,
        "notification_id": notification_id,
        "status": "QUEUED"
    }

In [ ]:
result_1 = create_notification(
    user_id="user_001",
    channel="email",
    message="Reset your password.",
    notification_type="transactional",
    idempotency_key="password-reset-001"
)
print("First request:", result_1)

result_2 = create_notification(
    user_id="user_001",
    channel="email",
    message="Reset your password.",
    notification_type="transactional",
    idempotency_key="password-reset-001"
)
print("Duplicate request:", result_2)

# 8. Extensible Notification Channels

All channels expose the same `send()` interface.

To add a future channel such as WhatsApp, implement the same interface and register it in the channel registry. The worker does not need to know provider-specific details.

In [ ]:
class NotificationChannel:
    def send(self, notification):
        raise NotImplementedError


def simulate_provider_response():
    # 80% success rate for demonstration.
    if random.random() < 0.8:
        return {"success": True, "message": "Notification sent."}

    return {
        "success": False,
        "message": "Third-party provider unavailable."
    }


class EmailChannel(NotificationChannel):
    def send(self, notification):
        print(f"Sending EMAIL to {notification['user_id']}")
        return simulate_provider_response()


class SMSChannel(NotificationChannel):
    def send(self, notification):
        print(f"Sending SMS to {notification['user_id']}")
        return simulate_provider_response()


class PushChannel(NotificationChannel):
    def send(self, notification):
        print(f"Sending PUSH notification to {notification['user_id']}")
        return simulate_provider_response()


channels = {
    "email": EmailChannel(),
    "sms": SMSChannel(),
    "push": PushChannel()
}

# 9. Delivery Status and Retry Logic

Possible states:

`QUEUED → SENT`

or, after temporary provider failures:

`QUEUED → FAILED ATTEMPT → RETRY → SENT`

After the maximum number of retries, the notification becomes `FAILED`.

The retry delay uses exponential backoff.

In [ ]:
MAX_RETRIES = 3


def record_attempt(notification_id, attempt_number, status, error_message=None):
    cursor.execute("""
    INSERT INTO delivery_attempts (
        id, notification_id, attempt_number,
        status, error_message, created_at
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        str(uuid.uuid4()),
        notification_id,
        attempt_number,
        status,
        error_message,
        datetime.utcnow().isoformat()
    ))
    connection.commit()


def update_notification_status(notification_id, status, retry_count=None):
    now = datetime.utcnow().isoformat()

    if retry_count is not None:
        cursor.execute("""
        UPDATE notifications
        SET status = ?, retry_count = ?, updated_at = ?
        WHERE id = ?
        """, (status, retry_count, now, notification_id))
    else:
        cursor.execute("""
        UPDATE notifications
        SET status = ?, updated_at = ?
        WHERE id = ?
        """, (status, now, notification_id))

    connection.commit()


def process_notification(notification):
    notification_id = notification["id"]
    channel = channels.get(notification["channel"])

    if channel is None:
        update_notification_status(notification_id, "FAILED")
        return

    for attempt in range(1, MAX_RETRIES + 1):
        response = channel.send(notification)

        if response["success"]:
            record_attempt(notification_id, attempt, "SENT")
            update_notification_status(
                notification_id, "SENT", attempt
            )
            print(f"Notification {notification_id} SENT.")
            return

        record_attempt(
            notification_id,
            attempt,
            "FAILED",
            response["message"]
        )

        print(f"Attempt {attempt} failed.")

        if attempt < MAX_RETRIES:
            # Exponential backoff.
            delay = 2 ** attempt
            print(f"Retrying in {delay} seconds...")
            time.sleep(delay)

    update_notification_status(
        notification_id,
        "FAILED",
        MAX_RETRIES
    )
    print(f"Notification {notification_id} permanently FAILED.")

# 10. Worker System

The API only accepts and queues notifications. Workers perform provider delivery asynchronously.

Transactional traffic is checked first so important messages are not blocked behind a large marketing campaign.

In [ ]:
def notification_worker():
    while True:
        try:
            if not transactional_queue.empty():
                _, notification = transactional_queue.get_nowait()
            else:
                _, notification = bulk_queue.get_nowait()

            process_notification(notification)

        except Empty:
            break


def run_workers():
    worker = threading.Thread(target=notification_worker)
    worker.start()
    worker.join()

    print("Worker finished processing.")

# 11. Flask REST API

In [ ]:
app = Flask(__name__)
CORS(app)


@app.route("/notifications", methods=["POST"])
def send_notification():
    data = request.get_json(silent=True) or {}

    required_fields = [
        "user_id",
        "channel",
        "message",
        "notification_type",
        "idempotency_key"
    ]

    missing = [field for field in required_fields if field not in data]

    if missing:
        return jsonify({
            "error": "Missing fields",
            "fields": missing
        }), 400

    result = create_notification(
        user_id=data["user_id"],
        channel=data["channel"],
        message=data["message"],
        notification_type=data["notification_type"],
        idempotency_key=data["idempotency_key"]
    )

    if "error" in result:
        return jsonify(result), 400

    return jsonify(result), 201


@app.route("/notifications", methods=["GET"])
def get_notifications():
    cursor.execute("""
    SELECT id, user_id, channel, message,
           notification_type, status,
           retry_count, created_at
    FROM notifications
    ORDER BY created_at DESC
    """)

    rows = cursor.fetchall()

    notifications = []

    for row in rows:
        notifications.append({
            "id": row[0],
            "user_id": row[1],
            "channel": row[2],
            "message": row[3],
            "notification_type": row[4],
            "status": row[5],
            "retry_count": row[6],
            "created_at": row[7]
        })

    return jsonify(notifications)

# 12. API Test

Flask's test client lets us exercise the API directly inside Colab without exposing a public web server.

In [ ]:
with app.test_client() as client:
    response = client.post(
        "/notifications",
        json={
            "user_id": "user_001",
            "channel": "push",
            "message": "Your payment was successful.",
            "notification_type": "transactional",
            "idempotency_key": "payment-001"
        }
    )

    print("HTTP status:", response.status_code)
    print(response.get_json())

# 13. HTML/CSS/JavaScript Dashboard

The following cell creates a standalone frontend.

It can be served alongside the Flask API in a normal local project. In Colab, the backend API can be tested with Flask's test client, while the HTML can be exported into the project repository.

In [ ]:
%%writefile index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Notification Service Dashboard</title>
<style>
body {
    font-family: Arial, sans-serif;
    background: #f4f6f8;
    margin: 0;
    padding: 40px;
}
.container {
    max-width: 900px;
    margin: auto;
}
.card {
    background: white;
    padding: 25px;
    margin-bottom: 25px;
    border-radius: 12px;
    box-shadow: 0 4px 16px rgba(0,0,0,.06);
}
input, select, textarea, button {
    width: 100%;
    box-sizing: border-box;
    padding: 11px;
    margin-top: 8px;
    margin-bottom: 15px;
}
textarea {
    min-height: 100px;
}
button {
    border: 0;
    border-radius: 6px;
    background: #2563eb;
    color: white;
    cursor: pointer;
}
.notification {
    padding: 14px 0;
    border-bottom: 1px solid #ddd;
}
</style>
</head>
<body>
<div class="container">
    <div class="card">
        <h1>Notification Service Dashboard</h1>
        <p>Prototype administration dashboard.</p>
    </div>

    <div class="card">
        <h2>Create Notification</h2>

        <input id="userId" placeholder="User ID">

        <select id="channel">
            <option value="email">Email</option>
            <option value="sms">SMS</option>
            <option value="push">Push</option>
        </select>

        <select id="type">
            <option value="transactional">Transactional</option>
            <option value="bulk">Bulk</option>
        </select>

        <textarea id="message" placeholder="Notification message"></textarea>

        <input id="idempotencyKey" placeholder="Idempotency Key">

        <button onclick="sendNotification()">
            Send Notification
        </button>
    </div>

    <div class="card">
        <h2>Notification Status</h2>
        <button onclick="loadNotifications()">Refresh</button>
        <div id="notifications"></div>
    </div>
</div>

<script>
const API_URL = "http://127.0.0.1:5000";

async function sendNotification() {
    const data = {
        user_id: document.getElementById("userId").value,
        channel: document.getElementById("channel").value,
        message: document.getElementById("message").value,
        notification_type: document.getElementById("type").value,
        idempotency_key: document.getElementById("idempotencyKey").value
    };

    const response = await fetch(API_URL + "/notifications", {
        method: "POST",
        headers: {"Content-Type": "application/json"},
        body: JSON.stringify(data)
    });

    const result = await response.json();
    alert(JSON.stringify(result, null, 2));
}

async function loadNotifications() {
    const response = await fetch(API_URL + "/notifications");
    const notifications = await response.json();

    const container = document.getElementById("notifications");
    container.innerHTML = "";

    notifications.forEach(notification => {
        const element = document.createElement("div");
        element.className = "notification";

        element.innerHTML = `
            <strong>${notification.channel.toUpperCase()}</strong>
            <p>${notification.message}</p>
            <p>Status: ${notification.status}</p>
            <small>Retries: ${notification.retry_count}</small>
        `;

        container.appendChild(element);
    });
}
</script>
</body>
</html>

# 14. End-to-End Demonstration

Create a transactional notification and a bulk notification, then run the worker.

The transactional notification is placed in the high-priority queue and should be processed before bulk work.

In [ ]:
# Create another user with all channels enabled.
create_user_preferences(
    user_id="user_002",
    email_enabled=True,
    sms_enabled=True,
    push_enabled=True
)

transactional = create_notification(
    user_id="user_002",
    channel="email",
    message="Your password was changed.",
    notification_type="transactional",
    idempotency_key="security-event-001"
)

bulk = create_notification(
    user_id="user_002",
    channel="push",
    message="Check out our latest offers.",
    notification_type="bulk",
    idempotency_key="campaign-001"
)

print("Transactional:", transactional)
print("Bulk:", bulk)

run_workers()

# 15. Verify Delivery Status

In [ ]:
with app.test_client() as client:
    response = client.get("/notifications")
    notifications = response.get_json()

    for notification in notifications:
        print(
            notification["id"],
            "|",
            notification["channel"],
            "|",
            notification["notification_type"],
            "|",
            notification["status"],
            "| retries:",
            notification["retry_count"]
        )

# 16. Failure Simulation

The provider simulator intentionally fails some requests.

This demonstrates that the notification service does not immediately give up when an external provider is temporarily unavailable.

Instead it:
1. Records the failed attempt.
2. Retries.
3. Uses exponential backoff.
4. Eventually marks the notification as FAILED if all attempts fail.

# 17. Production Scaling

The notebook prototype is intentionally simple.

At approximately 10 million notifications/day, the production design should use:

```text
Load Balancer
      |
      v
Multiple Notification API instances
      |
      v
Durable Database
      |
      v
Kafka / RabbitMQ / SQS
      |
      +-------------------+
      |                   |
Transactional         Bulk
Workers               Workers
      |                   |
      +---------+---------+
                |
        Provider Adapters
       /        |         \
    Email      SMS        Push
```

Scale worker groups independently according to queue depth and provider capacity.

# 18. Important Reliability Tradeoffs

### At-least-once delivery
Messages are retried rather than silently discarded. This improves reliability but means duplicate processing is possible.

### Idempotency
A stable idempotency key prevents repeated requests from creating duplicate notification records.

### Priority separation
Transactional and bulk workloads use separate queues so a marketing blast cannot overwhelm password resets or security alerts.

### Provider failures
Retries with exponential backoff handle temporary failures and reduce pressure on rate-limited providers.

### Database and queue durability
The prototype uses SQLite and in-memory queues for simplicity. Production should use durable, replicated infrastructure.

### Extensibility
Channel-specific provider code is isolated behind a common interface so new channels can be added without rewriting the core notification service.

# 19. Future Improvements

1. Replace SQLite with PostgreSQL.
2. Replace in-memory queues with Kafka, RabbitMQ, or SQS.
3. Add real provider adapters.
4. Add provider-specific rate limiting.
5. Add circuit breakers.
6. Add dead-letter queues.
7. Implement full quiet-hour evaluation.
8. Add delivery webhooks for SENT/DELIVERED/FAILED updates.
9. Add authentication and authorization to the API.
10. Add metrics, logs, tracing, and alerting.
11. Add horizontal worker scaling.
12. Add automated integration and load tests.

# 20. Conclusion

This prototype demonstrates the core architecture of a scalable notification service.

It supports Email, SMS and Push through interchangeable channel implementations, checks user channel preferences, uses idempotency for de-duplication, tracks delivery attempts and status, separates transactional and bulk traffic, and retries unreliable provider calls.

For production, the main architectural change would be replacing the notebook's local SQLite database, in-memory queues, and threads with durable distributed infrastructure and horizontally scalable workers.